In [1]:
from torchvision.datasets import MNIST, FashionMNIST, CIFAR10
import torchvision
import numpy as np
import random

import torch
import torch.nn.functional as F
import cl_gym as cl

import sys
import os

init_path = os.path.abspath('.')
new_path = init_path
while True:
    if new_path[-3:] == "FSW":
        sys.path.append(new_path)
        break
    new_path = os.path.abspath('..')
    os.chdir(new_path)


seed = 0

np.random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = False
torch.set_num_threads(8)

def make_params() -> dict:
    import os
    from pathlib import Path
    import uuid

    params = {
            # dataset
            'dataset': "FashionMNIST",
            'fairness_agg': 'mean',
            'model': 'MLP',

            # benchmark
            'seed': seed,
            'num_tasks': 5,
            'epochs_per_task': 5,
            'per_task_examples': np.inf,
            # 'per_task_examples': 10000,
            'per_task_memory_examples': 64,
            'batch_size_train': 64,
            'batch_size_memory': 64,
            'batch_size_validation': 256,
            'tau': 10.0,

            # algorithm
            'optimizer': 'sgd',
            'learning_rate': 0.001,
            'momentum': 0.9,
            'learning_rate_decay': 1.0,
            'criterion': torch.nn.CrossEntropyLoss(),
            # 'criterion': torch.nn.BCEWithLogitsLoss(),

            'device': torch.device('cuda:7' if torch.cuda.is_available() else 'cpu'),
             
            # sample selection
            'alpha': 0.001,
            'metric' : "EER",
            'lambda': 0.1,
            'lambda_old': 0.0,

            # postprocessing
            # "post_processing": "eps_fairness"

              }
    

#     trial_id = str(uuid.uuid4())
    trial_id = f"demo/dataset={params['dataset']}/seed={params['seed']}_epoch={params['epochs_per_task']}_lr={params['learning_rate']}_tau={params['tau']}_alpha={params['alpha']}"
    if params['lambda'] != 0:
        trial_id+=f"_lmbd_{params['lambda']}_lmbdold_{params['lambda_old']}"
    params['trial_id'] = trial_id
    params['output_dir'] = os.path.join("./outputs/{}".format(trial_id))
    print(f"output_dir={params['output_dir']}")
    Path(params['output_dir']).mkdir(parents=True, exist_ok=True)

    return params

params = make_params()

output_dir=./outputs/demo/dataset=FashionMNIST/seed=0_epoch=5_lr=0.001_tau=10.0_alpha=0.001_lmbd_0.1_lmbdold_0.0


In [2]:
"MNIST" in params['dataset']

True

In [3]:
from datasets import MNIST
from datasets import FashionMNIST

if params['dataset'] == 'MNIST':
    benchmark = MNIST(num_tasks=params['num_tasks'],
                    per_task_memory_examples=params['per_task_memory_examples'],
                    per_task_examples = params['per_task_examples'],
                    random_class_idx = False)
    input_dim = (28, 28)
elif params['dataset'] == 'FashionMNIST':
    benchmark = FashionMNIST(num_tasks=params['num_tasks'],
                            per_task_memory_examples=params['per_task_memory_examples'],
                            per_task_examples = params['per_task_examples'],
                    random_class_idx = False)
    input_dim = (28, 28)
else:
    raise NotImplementedError
class_idx = benchmark.class_idx
num_classes = len(class_idx)



[0 1 2 3 4 5 6 7 8 9]


In [4]:
from algorithms.imbalance import Heuristic2
from metrics import MetricCollector2
from backbones import MLP2Layers2
from trainers.imbalance_trainer import ImbalanceContinualTrainer1 as ContinualTrainer


backbone = MLP2Layers2(
    input_dim=input_dim, 
    hidden_dim_1=256, 
    hidden_dim_2=256, 
    output_dim=num_classes,
    class_idx=class_idx,
    config=params
    ).to(params['device'])

algorithm = Heuristic2(backbone, benchmark, params, requires_memory=True)

metric_manager_callback = MetricCollector2(num_tasks=params['num_tasks'],
                                                        eval_interval='epoch',
                                                        epochs_per_task=params['epochs_per_task'])

trainer = ContinualTrainer(algorithm, params, callbacks=[metric_manager_callback])


In [5]:
trainer.run()
print("final avg-acc", metric_manager_callback.meters['accuracy'].compute_final())
print("final avg-forget", metric_manager_callback.meters['forgetting'].compute_final())

---------------------------- Task 1 -----------------------
[1] Eval metrics for task 1 >> {'accuracy': 0.9715, 'loss': 0.00033985645323991774, 'std': 0.004500000000000004, 'EER': -1}
[2] Eval metrics for task 1 >> {'accuracy': 0.978, 'loss': 0.00024415119364857673, 'std': 0.0, 'EER': -1}
[3] Eval metrics for task 1 >> {'accuracy': 0.9815, 'loss': 0.00020586757734417915, 'std': 0.006500000000000006, 'EER': -1}
[4] Eval metrics for task 1 >> {'accuracy': 0.984, 'loss': 0.00018824721965938807, 'std': 0.007000000000000006, 'EER': -1}
[5] Eval metrics for task 1 >> {'accuracy': 0.985, 'loss': 0.00017593078874051572, 'std': 0.006000000000000005, 'EER': -1}
training_task_end
---------------------------- Task 2 -----------------------
loss_group=tensor([[ 0.0198,  0.0195,  8.7685, 12.5658]])
Elapsed time(grad):2.753
### Cplex absolute_and_nonabsolute_minsum LP solver ###
Elapsed time(optim):5.365
Fairness:[ 0.          0.79366163  0.06358232 -0.85724415]
Current class expected loss:tensor([[ 

In [ ]:
import copy
task_weight = copy.deepcopy(algorithm.weight_all)

num_bin = 20
np.arange(0+1/num_bin, 1+1/num_bin, 1/num_bin)

def bin(w: np.array, num_bin=20):
    out = dict()
    for r in np.arange(0+1/num_bin, 1+1/num_bin, 1/num_bin):
        r = np.round(r, 2)
        out[r] = np.sum(np.logical_and(w<=r, r-1/num_bin<w))
    out[1/num_bin] += np.sum(w==0)
    kk = list(out.keys())
    for k in kk:
        if out[k] == 0:
            del(out[k])
    return out


binned_weight = dict()
for i, wt in enumerate(task_weight):
    if i==0:
        continue
    print(f"task:{i+1}")
    binned_weight[i+1] = list()
    for we in wt:
        binned_weight[i+1].append({k: bin(we[k]) for k in we})




task:2
task:3
task:4
task:5


In [7]:
binned_weight

{2: [{2: {0.05: 223, 0.15: 1, 1.0: 5776}, 3: {1.0: 6000}},
  {2: {0.05: 5200, 0.45: 1, 1.0: 799},
   3: {0.05: 5215, 0.15: 1, 0.25: 1, 1.0: 783}},
  {2: {0.05: 5203, 0.5: 1, 1.0: 796},
   3: {0.05: 5381, 0.1: 1, 0.45: 1, 1.0: 617}},
  {2: {0.05: 5575, 0.4: 1, 1.0: 424}, 3: {0.05: 5303, 0.2: 1, 1.0: 696}},
  {2: {0.05: 5308, 0.9: 1, 1.0: 691},
   3: {0.05: 5284, 0.25: 1, 0.7: 1, 1.0: 714}}],
 3: [{4: {1.0: 6000}, 5: {0.05: 3011, 0.75: 1, 1.0: 2988}},
  {4: {0.05: 5575, 0.55: 1, 1.0: 424}, 5: {0.05: 5846, 0.8: 1, 1.0: 153}},
  {4: {0.05: 5381, 0.25: 1, 0.35: 1, 1.0: 617},
   5: {0.05: 5879, 0.15: 1, 1.0: 120}},
  {4: {0.05: 5382, 0.1: 1, 1.0: 617}, 5: {0.05: 5820, 0.85: 1, 1.0: 179}},
  {4: {0.05: 5294, 0.1: 1, 0.95: 1, 1.0: 704},
   5: {0.05: 5821, 0.1: 1, 1.0: 178}}],
 4: [{6: {1.0: 6000}, 7: {1.0: 6000}},
  {6: {0.05: 5654, 0.1: 1, 0.65: 1, 0.7: 1, 1.0: 343},
   7: {0.05: 5883, 1.0: 117}},
  {6: {0.05: 5609, 0.25: 1, 0.35: 1, 0.65: 1, 0.8: 1, 1.0: 387},
   7: {0.05: 6000}},
  {6: {0.0

In [8]:
metric_manager_callback.meters['accuracy'].get_data()

array([[0.985, 0.   , 0.   , 0.   , 0.   ],
       [0.941, 0.773, 0.   , 0.   , 0.   ],
       [0.829, 0.817, 0.838, 0.   , 0.   ],
       [0.784, 0.662, 0.622, 0.753, 0.   ],
       [0.772, 0.607, 0.63 , 0.822, 0.85 ]])

In [9]:
np.mean(metric_manager_callback.meters['accuracy'].compute_overall())

0.8224083333333334

In [10]:
[np.round(x, 3) for x in metric_manager_callback.meters['EER'].compute_overall()]

[0.0, 0.084, 0.007, 0.063, 0.094]

In [11]:
np.mean(metric_manager_callback.meters['EER'].compute_overall())

0.049843888888888896